# Intro to Cognee — building a Company Brain

## Why a company brain?

Company knowledge is **fragmented by default**. The same context is smeared across
Slack threads, meeting notes, email, docs, and tickets — and no single person has
seen all of it. A bug gets reported in one channel, diagnosed in a second, and
resolved in a meeting nobody else attended. Three months later someone re-asks the
same question because the answer is unfindable.

Meanwhile everyone has wired up their own AI tools — but each one is an island. Your
chatbot knows your prompt, not last week's standup; your meeting notetaker captures
the call, then forgets it. The intelligence is there; the **shared memory** is not.

A *company brain* fixes the memory, not the model: ingest the places people actually
talk into **one knowledge graph**, where people, projects, topics, decisions, and
issues are linked across every source. Then point applications at that graph.

## What one shared graph unlocks

Once the context is unified and queryable, the same graph powers a family of apps:

- **Support agent for repeat questions** — across many client channels, customers
  ask the same things. The graph already holds the prior answer, so an agent can
  respond instead of pinging a human again.
- **Expert finder** — *who* should I nudge about this? The graph knows who has
  spoken to a topic before, so questions route to the right person automatically.
- **Project follow-up agent** — from deadlines, emails, and meeting notes, remind the
  relevant people when an action is due or slipping.
- **Catch-up / onboarding search** — missed a meeting, or not staffed on a project but
  need to know where it stands? Ask the graph instead of reconstructing it by hand.
- **Hierarchy-aware disambiguation** — when facts conflict, resolve them using the
  company's structure (a manager's statement outranks a junior's guess).
- **Permissions across all of the above** — the graph is scoped so each person and
  agent only sees what they're allowed to. (Cognee tags every node with a node set —
  the same mechanism we use in section 3 — which is the foundation access control
  builds on.)

This notebook builds the **foundation** those apps stand on: the unified graph, and
the core `remember → recall → improve` loop. We use a tiny slice of real data — a few
Slack threads and a Granola meeting note — so you can see each idea clearly:

1. **Load data** — `remember()` ingests text *and* builds the graph in one call
2. **Recall** — ask the graph natural-language questions
3. **Node sets** — tag data so you can scope recall to a source/person/project
4. **Custom graph model** — replace generic extraction with a typed schema
5. **Improve** — feedback and self-improvement make the graph sharper over time
6. **Add another source** — drop Granola notes into the same graph

## The API (Cognee 1.x)

The modern surface is three async functions:

```python
await cognee.remember(text, dataset_name=...)    # 1. ingest + build the graph
await cognee.recall(query)                       # 2. ask questions
await cognee.improve(dataset=...)                # 3. refine the graph from feedback/use
```

**`remember()` replaces the old two-step `add()` + `cognify()`** — it ingests the
text, extracts entities/relationships into the graph, *and* (by default) runs a
self-improvement pass. You rarely call `cognify()` directly anymore.

> **Where did `cognify` go?** It still exists. `remember()` = `add()` +
> `cognify()` + self-improvement rolled into one. Reach for the lower-level
> `cognee.add()` + `cognee.cognify(...)` only when you want to **decouple
> ingestion from graph-building** — e.g. bulk-load hundreds of docs and build
> the graph once, re-process existing data with a different schema without
> re-uploading, or run the build as a background job. For everyday use,
> `remember()` is the one call you need.

**The improvement piece.** A company brain is never "done" — new conversations
arrive and earlier answers get corrected. Cognee bakes this in:
`remember(..., self_improvement=True)` (the default) refines the graph on every
write, `cognee.improve()` runs an explicit enrichment pass, and
`recall(..., feedback_influence=...)` lets prior feedback steer future answers.
That's what makes it a *living* graph rather than a one-shot index.

## Setup

You need an LLM key. Put `LLM_API_KEY=sk-...` in the repo's `.env` (Cognee uses
it for extraction and recall). We turn session caching **on** (`CACHING=true`)
because the feedback section (5) needs session memory.

> Run once: `uv pip install -e ".[dev]" jupyterlab ipykernel`

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")   # picks up LLM_API_KEY
os.environ.setdefault("CACHING", "true")  # session memory ON (needed for the feedback section)
os.environ.setdefault("LLM_API_KEY", "your-api-key")

import cognee
from cognee.modules.users.methods import get_default_user
from cognee.modules.data.methods import get_authorized_existing_datasets
from cognee.context_global_variables import set_database_global_context_variables

# sample_data lives at the repo root; handle running from tutorial/ or root
ROOT = Path.cwd()
DATA = ROOT / "sample_data" if (ROOT / "sample_data").exists() else ROOT.parent / "sample_data"
DATASET = "company_brain"

async def reset():
    """Wipe the graph so each section starts from a clean slate."""
    await cognee.prune.prune_data()
    await cognee.prune.prune_system(metadata=True)

import re as _re  # (re already imported above)
def first_speaker(text):
    m = _re.search(r"\[([^,\]]+@[^,\]]+),", text)
    return m.group(1) if m else "unknown"

def answer(results):
    """Pull the natural-language answer out of a recall() result list.
    recall() can return a graph entry (.text), a QA entry (.answer), or a
    context entry (.content) depending on how the query auto-routes."""
    for r in results:
        for attr in ("text", "answer", "content"):
            value = getattr(r, attr, None)
            if value:
                return value
    return str(results[0]) if results else "<no answer>"


async def show_graph(filename, dataset, height=500):
    """Render a dataset-scoped graph to HTML and embed it."""
    user = await get_default_user()
    datasets = await get_authorized_existing_datasets([dataset], "read", user)

    if not datasets:
        raise ValueError(f"Dataset not found or not readable: {dataset_name}")

    dataset = datasets[0]
    owner_id = getattr(dataset, "owner_id", None) or user.id

    async with set_database_global_context_variables(dataset.id, owner_id):
        await cognee.visualize_graph(str(ROOT / filename))
    from IPython.display import IFrame
    return IFrame(src=filename, width="100%", height=height)


from cognee.api.v1.session import add_feedback, get_session

async def show_session(session_id):
    """Print what's stored in session memory for each Q&A turn."""
    entries = await get_session(session_id)
    if not entries:
        print("  (session empty)"); return
    for e in entries:
        print(f"  Q: {(e.question or '')[:55]}")
        print(f"     feedback_text : {e.feedback_text}")
        print(f"     feedback_score: {e.feedback_score}")
        print(f"     memify_metadata: {e.memify_metadata}")

async def show_node_weights(used_graph_element_ids, limit=6):
    """Print cognee's feedback weights for the graph nodes an answer leaned on."""
    from cognee.infrastructure.databases.graph import get_graph_engine
    node_ids = (used_graph_element_ids or {}).get("node_ids", [])
    g = await get_graph_engine()
    weights = await g.get_node_feedback_weights(node_ids)
    return {k: weights[k] for k in list(weights)[:limit]}

print("sample data:", sorted(p.name for p in DATA.glob("*.txt")))


2026-06-08T12:35:18.703158 [info     ] Deleted old log file: /Users/milenko/.cognee/logs/2026-06-08_12-19-10.log [cognee.shared.logging_utils]

2026-06-08T12:35:18.703530 [info     ] Log file created at: /Users/milenko/.cognee/logs/2026-06-08_14-35-18.log [cognee.shared.logging_utils] log_file=/Users/milenko/.cognee/logs/2026-06-08_14-35-18.log

2026-06-08T12:35:18.703753 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.logging_utils]

2026-06-08T12:35:18.703939 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_version=1.1.2 database_path=/Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databa

sample data: ['granola.txt', 'slack_1.txt', 'slack_2.txt', 'slack_3.txt']


## 1. Load the Slack data with `remember()`

Our Slack threads are saved as plain `.txt` transcripts in `sample_data/` — one
line per message, `[speaker, timestamp] text`. `remember()` takes raw text,
ingests it, and builds the knowledge graph in a single call.

> **In production** you wouldn't read files — you'd pull live threads from the
> Slack API and normalize them to this same shape (see
> `src/company_brain/sources/slack.py`). Once text is in this shape, the source
> doesn't matter.

In [2]:
# Peek at the raw input first. Every source — Slack here, Granola later — is
# normalized to the same transcript shape: one line per message,
# "[speaker, timestamp] text". This is all cognee needs.
slack_files = sorted(DATA.glob("slack_*.txt"))
print(slack_files[1].read_text())   # slack_2: the node-sets bug report

# #brain-project: i noticed that there is a bug in code when you want to call remember with node s
[veljko@topoteretes.com, 2026-05-29T09:02] i noticed that there is a bug in code when you want to call remember with node sets. It keeps extracting everything. Do you have any idea what is it about?
## Reported issue candidates
- [veljko@topoteretes.com, 2026-05-29T09:02] i noticed that there is a bug in code when you want to call remember with node sets. It keeps extracting everything. Do you have any idea what is it about?


In [3]:
await reset()

slack_files = sorted(DATA.glob("slack_*.txt"))
for f in slack_files:
    result = await cognee.remember(f.read_text(), dataset_name=DATASET)
    print("remembered", f.name, "— status:", result.to_dict().get("status"))


2026-06-08T12:35:20.351185 [info     ] Deleted Ladybug database files at /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/ef60b404-ebd9-4b03-ae4d-f30bfcfe5702/954a678d-fe49-50a6-bcf0-76cf1e8c7977.lbug [cognee.shared.logging_utils]

2026-06-08T12:35:20.360022 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:35:20.466706 [info     ] Deleted Ladybug database files at /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/ef60b404-ebd9-4b03-ae4d-f30bfcfe5702/1a68d6ef-c9d8-5e77-8360-57dd3dd8bac3.lbug [cognee.shared.logging_utils]

2026-06-08T12:35:20.475376 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:35:20.530998 [info     ] Log file created at: /Users/milenko/.cognee/logs/2026-06-08_14-35-18.log [cognee.shared.logging_utils] log_file=/Users/milenko/.cognee/logs/20

remembered slack_1.txt — status: completed



2026-06-08T12:36:07.778145 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:36:07.779028 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-08T12:36:07.839885 [info     ] Pipeline run started: `a27e219b-a36c-5691-9a19-626c40b4d2b7` [run_tasks_with_telemetry()]

2026-06-08T12:36:07.846868 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-08T12:36:07.854304 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-08T12:36:07.866014 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-08T12:36:42.353973 [info     ] No close match found for 'person' in category 'classes' [OntologyAdapter]

2026-06-08T12:36:42.354605 [info     ] No close match found for 'veljko@topoteretes.com' in category 'individuals' [OntologyAdapter]

2026-06-08T12:36:42.354949 [info     ] No close ma

remembered slack_2.txt — status: completed



2026-06-08T12:36:44.707667 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:36:44.708416 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-08T12:36:44.765765 [info     ] Pipeline run started: `a27e219b-a36c-5691-9a19-626c40b4d2b7` [run_tasks_with_telemetry()]

2026-06-08T12:36:44.772096 [info     ] Coroutine task started: `classify_documents` [run_tasks_base]

2026-06-08T12:36:44.778272 [info     ] Async Generator task started: `extract_chunks_from_documents` [run_tasks_base]

2026-06-08T12:36:44.786962 [info     ] Coroutine task started: `extract_graph_and_summarize` [run_tasks_base]

2026-06-08T12:37:28.024807 [info     ] No close match found for 'project' in category 'classes' [OntologyAdapter]

2026-06-08T12:37:28.025434 [info     ] No close match found for 'brain-project' in category 'individuals' [OntologyAdapter]

2026-06-08T12:37:28.025758 [info     ] No close match foun

remembered slack_3.txt — status: completed


## 2. Recall — ask the graph

`recall()` runs a graph-aware search and returns a natural-language answer by
default (it auto-routes the query for you). Pass `only_context=True` to get the
raw retrieved context instead of a synthesized answer — useful for seeing *what*
the graph pulled before it's summarized.

Our data: someone reported a bug about `remember` with node sets. Let's ask.

In [4]:
results = await cognee.recall(
    "What bug was reported, and who reported it?",
    datasets=[DATASET],
)
print(answer(results))


2026-06-08T12:37:29.950722 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='What bug was reported, and who reported it?' [query_router]

2026-06-08T12:37:30.757349 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.70s [cognee.shared.logging_utils]

2026-06-08T12:37:30.757974 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-08T12:37:30.763683 [info     ] ID-filtered retrieval: 29 nodes and 52 edges in 0.01s [cognee.shared.logging_utils]

2026-06-08T12:37:30.764297 [info     ] Graph projection completed: 29 nodes, 52 edges in 0.00s [CogneeGraph]

2026-06-08T12:37:30.764903 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 11, 'connection_count': 10}

2026-06-08T12:37:34.183842 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:37:34.242263 [info     ] recall: 1 results across sources=['graph'] (

Bug: calling the remember function with node sets causes it to extract everything. Reported by: veljko@topoteretes.com (reported 2026-05-29T09:02).


In [5]:
# Peek under the hood: the raw context the answer was built from
ctx = await cognee.recall(
    "node sets bug",
    datasets=[DATASET],
    only_context=True,
    top_k=3,
)
for c in ctx:
    print("-", str(getattr(c, "content", c))[:200])


2026-06-08T12:37:34.254762 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='node sets bug' [query_router]

2026-06-08T12:37:35.158890 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.79s [cognee.shared.logging_utils]

2026-06-08T12:37:35.159571 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-08T12:37:35.165537 [info     ] ID-filtered retrieval: 29 nodes and 52 edges in 0.01s [cognee.shared.logging_utils]

2026-06-08T12:37:35.166142 [info     ] Graph projection completed: 29 nodes, 52 edges in 0.00s [CogneeGraph]

2026-06-08T12:37:35.166675 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 4, 'connection_count': 3}

2026-06-08T12:37:35.196192 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:37:35.236601 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


- kind='graph_completion' search_type='GRAPH_COMPLETION' text='Nodes:\nNode: bug: remember with node sets extracts everything\n__node_content_start__\nBug reported in code where calling remember with no


## 3. Node sets — tag data so you can scope and permission it

**Node sets** are tags you attach at ingest time — `source:slack`, `project:x`,
`speaker:y`. They're how you later **scope** a query ("only answer from Slack")
and how you build **permissions** ("this user only sees these node sets").

You attach them by passing `node_set=[...]`, and you scope a recall with
`node_name=[...]`:

```python
await cognee.remember(text, dataset_name=ds, node_set=["source:slack"])
await cognee.recall(query, datasets=[ds], node_name=["source:slack"], scope=["graph"])
```

> **Heads-up for this demo:** scope **enforcement** is reliable on the Cognee
> server/cloud (where the bot in `BOT.md` runs). In this lightweight *in-process*
> notebook the recall-side filter is inconsistent, so don't be surprised if a
> scoped query here still pulls in everything. The tagging API below is exactly
> what you use in production.

In [6]:
# Tag a couple of sources as we ingest them.
NS_DATASET = "node_sets_demo"
for f in slack_files:
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:slack"])
for f in sorted(DATA.glob("granola*.txt")):
    await cognee.remember(f.read_text(), dataset_name=NS_DATASET, node_set=["source:granola"])
print("tagged Slack as source:slack and the meeting note as source:granola")


2026-06-08T12:37:35.294473 [info     ] Pipeline run started: `7cc20b9d-6c53-56c2-9b32-734f9122f86d` [run_tasks_with_telemetry()]

2026-06-08T12:37:35.300694 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-06-08T12:37:35.306966 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-06-08T12:37:35.324391 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-06-08T12:37:35.330632 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-06-08T12:37:35.336723 [info     ] Pipeline run completed: `7cc20b9d-6c53-56c2-9b32-734f9122f86d` [run_tasks_with_telemetry()]

2026-06-08T12:37:35.471062 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:37:35.471857 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-08T12:37:35.529598 [info     ] Pipeline run started: `623a70f8-1a25-5bc4-b0b0-0c

tagged Slack as source:slack and the meeting note as source:granola


In [7]:
# Scope a recall to one node set. (Reliable on the server; in-process this is
# the API shape rather than a hard guarantee — see the note above.)
scoped = await cognee.recall(
    "What was discussed about node sets?",
    datasets=[NS_DATASET],
    node_name=["source:slack"],
    scope=["graph"],
)
print(answer(scoped))


2026-06-08T12:40:44.765461 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='What was discussed about node sets?' [query_router]

2026-06-08T12:40:45.601630 [info     ] Vector collection retrieval completed: Retrieved distances from 3 collections in 0.74s [cognee.shared.logging_utils]

2026-06-08T12:40:45.602129 [info     ] Retrieving graph filtered by node type and node name (NodeSet). [CogneeGraph]

2026-06-08T12:40:45.611440 [info     ] Graph projection completed: 17 nodes, 92 edges in 0.00s [CogneeGraph]

2026-06-08T12:40:45.611934 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 5, 'connection_count': 10}

2026-06-08T12:40:51.027598 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:40:51.072287 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


Node sets were mentioned as the parameter type passed to remember; calling remember with node sets triggers a bug that “keeps extracting everything.”


## 4. Custom graph model — extract a typed schema

Default extraction discovers entity types on its own, which can be noisy. When
you know the shape of your domain, give Cognee a **graph model**: a set of
Pydantic `DataPoint` classes. Extraction is then constrained to *your* types —
`Person`, `Message`, `Topic`, `Issue` — with the relationships you defined. Pair
it with a `custom_prompt` that tells the LLM how to fill the schema.

`remember()` accepts both, so this stays a one-call flow. (This is also the case
where you might instead split into `add()` + `cognify(graph_model=...)` if you
wanted to re-process existing data with a new schema without re-uploading it.)

Watch the two graphs below: the **before** is what default extraction
produced (lots of generic, self-discovered types), and the **after** is the
same conversations collapsed onto your `Person` / `Message` / `Issue` schema.

In [8]:
# BEFORE: the graph from default extraction (generic, self-discovered types)
await show_graph("graph_default.html", DATASET)


2026-06-08T12:40:51.180472 [info     ] Retrieved 29 nodes and 52 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-08T12:40:51.183476 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/graph_default.html [cognee.shared.logging_utils]

2026-06-08T12:40:51.183735 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/graph_default.html [cognee.shared.logging_utils]

2026-06-08T12:40:51.211458 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


In [9]:
from typing import Optional
from cognee.low_level import DataPoint

class Person(DataPoint):
    email: str
    name: str
    metadata: dict = {"index_fields": ["email", "name"], "identity_fields": ["email"]}

class Topic(DataPoint):
    label: str
    metadata: dict = {"index_fields": ["label"]}

class Issue(DataPoint):
    summary: str
    reported_by: Person
    metadata: dict = {"index_fields": ["summary"]}

class Message(DataPoint):
    speaker: Person
    text: str
    sent_at: str
    about: Optional[list[Topic]] = None
    reports: Optional[list[Issue]] = None
    metadata: dict = {"index_fields": ["text"]}

class Thread(DataPoint):
    channel: str
    started_at: str
    participants: list[Person]
    messages: list[Message]
    metadata: dict = {"index_fields": ["channel"]}

In [10]:
EXTRACTION_PROMPT = """Extract a company conversation graph from the transcript.
Use the provided Thread graph model with Person, Message, Topic, and Issue nodes.
- Keep every transcript line as a Message with its exact speaker, timestamp, and text.
- Create one Person per email and reuse the same Person across messages.
- If a message reports a bug or broken behavior, create an Issue linked from that Message.
Prefer exact source text over summaries."""

await reset()
for f in slack_files:
    text = f.read_text()
    await cognee.remember(
        text,
        dataset_name=DATASET,
        node_set=["source:slack", f"speaker:{first_speaker(text)}"],
        graph_model=Thread,            # <-- typed schema
        custom_prompt=EXTRACTION_PROMPT,
    )
print("typed graph built")


2026-06-08T12:40:51.384096 [info     ] Deleted Ladybug database files at /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/52218681-0263-423b-944a-7bc032c52281/83b2a159-66a6-5dc8-bebc-d75fe94820d8.lbug [cognee.shared.logging_utils]

2026-06-08T12:40:51.393101 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:40:51.499327 [info     ] Deleted Ladybug database files at /Users/milenko/cognee/cognee-companybrain-hackathon/.venv/lib/python3.13/site-packages/cognee/.cognee_system/databases/52218681-0263-423b-944a-7bc032c52281/57e95ce3-16c7-5511-b234-9b34905bc0fc.lbug [cognee.shared.logging_utils]

2026-06-08T12:40:51.507491 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:40:54.382465 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 

User 1c7d9524-27

typed graph built


In [11]:
# AFTER: the same data, now constrained to the typed Person/Message/Issue schema
await show_graph("graph_typed.html", DATASET)


2026-06-08T12:43:35.410212 [info     ] Retrieved 31 nodes and 46 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-08T12:43:35.413355 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/graph_typed.html [cognee.shared.logging_utils]

2026-06-08T12:43:35.413623 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/graph_typed.html [cognee.shared.logging_utils]

2026-06-08T12:43:35.442042 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


In [12]:
typed = await cognee.recall(
    "Summarize the reported issue and who is involved.",
    datasets=[DATASET],
)
print(answer(typed))


2026-06-08T12:43:35.456466 [info     ] query_router: routed=GRAPH_SUMMARY_COMPLETION score=5.0 query='Summarize the reported issue and who is involved.' scores={'GRAPH_SUMMARY_COMPLETION': 5.0} [query_router]

2026-06-08T12:43:36.305223 [info     ] Vector collection retrieval completed: Retrieved distances from 13 collections in 0.75s [cognee.shared.logging_utils]

2026-06-08T12:43:36.305865 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-08T12:43:36.311784 [info     ] ID-filtered retrieval: 31 nodes and 46 edges in 0.01s [cognee.shared.logging_utils]

2026-06-08T12:43:36.312400 [info     ] Graph projection completed: 31 nodes, 46 edges in 0.00s [CogneeGraph]

2026-06-08T12:43:36.312959 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 13, 'connection_count': 10}

2026-06-08T12:44:09.695825 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:44:09.743461 [info     ] 

Issue: A bug in the code where calling remember with node sets causes it to extract everything. Reporter: Veljko (veljko@topoteretes.com). Responder: Milenko (milenko@topoteretes.com) acknowledged the report and proposed a follow-up call. Context: #brain-project.


## 5. Feedback & self-improvement — and what changes in each store

A company brain should *learn*. Cognee improves in two ways:

- **Implicit** — `remember(..., self_improvement=True)` (the default) refines
  the graph on every write. You've had this in every cell above.
- **Explicit feedback loop** — `recall` → `add_feedback` → `improve` → `recall`.
  This teaches the brain things it can't extract from text, like *who owns what*.

We'll run that loop and **watch both stores change** at each step:

| Store | Inspect with | What `improve()` does to it |
|---|---|---|
| Session memory | `get_session()` | records your feedback, then flips `feedback_weights_applied` to `True` |
| Graph | `get_node_feedback_weights()` | shifts the feedback weight on the nodes the answer used |

Feedback needs session memory, so the setup cell sets `CACHING=true` and we pass
a `session_id` to `recall`. Scores are **1 (bad) … 5 (great)**.

In [62]:
# BEAT 0 — a small, self-contained graph for this lesson, then snapshot BOTH stores.
# (Default extraction in its own dataset, so the answer is concrete and node weights
#  start at the default 0.5 — independent of the typed graph above.)
FB_DATASET = "feedback_demo"
SESSION = "feedback-demo"
await reset()
from cognee.modules.engine.operations.setup import setup
await setup()
# Delete old cached Q&A/session state for this SESSION id.
from cognee.modules.users.methods import get_default_user
from cognee.infrastructure.session.get_session_manager import get_session_manager

user = await get_default_user()
await get_session_manager().delete_session(
    user_id=str(user.id),
    session_id=SESSION,
)


for f in slack_files:
    await cognee.remember(f.read_text(), dataset_name=FB_DATASET)

Q = "Who should I message next about the node sets bug?"
before = await cognee.recall(Q, datasets=[FB_DATASET], session_id=SESSION)
print("ANSWER (before):", answer(before))

qa = (await get_session(SESSION))[-1]          # the Q&A we just created
used = qa.used_graph_element_ids               # graph nodes/edges this answer used

print("\n— SESSION MEMORY (fresh turn) —");      await show_session(SESSION)
print("\n— GRAPH weights BEFORE —", await show_node_weights(used))


2026-06-08T14:13:37.628086 [info     ] Database deleted successfully. [cognee.shared.logging_utils]

Deleting cache...             

✓ Cache deleted successfully! 

User 6ffb96d2-7c5d-4bb6-8fa3-4bf5a2d6dab5 has registered.

2026-06-08T14:13:37.791635 [info     ] Pipeline run started: `10b70468-844c-5b4d-aee3-db35161921e5` [run_tasks_with_telemetry()]

2026-06-08T14:13:37.797900 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-06-08T14:13:37.804053 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-06-08T14:13:37.824798 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-06-08T14:13:37.831091 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-06-08T14:13:37.837700 [info     ] Pipeline run completed: `10b70468-844c-5b4d-aee3-db35161921e5` [run_tasks_with_telemetry()]

2026-06-08T14:13:38.012268 [info     ] Ladybug database closed successfully [cognee.shared.logging_util

ANSWER (before): Message Veljko — veljko@topoteretes.com.

— SESSION MEMORY (fresh turn) —
  Q: Who should I message next about the node sets bug?
     feedback_text : None
     feedback_score: None
     memify_metadata: None

— GRAPH weights BEFORE — {'4d59c4c0-2fb5-5aa0-b756-56e6442f8bbd': 0.5, '1aad5e98-0b66-505d-ad42-1f3282ee478c': 0.5, 'dffcc39f-2b68-59bc-962f-7dcfa5c2c9ba': 0.5, '14e8c4bb-ddcd-5a36-9732-54ee766711ad': 0.5, '073a334a-c403-5b35-ba66-a44d3f5d5774': 0.5, 'e55af088-7425-5cb1-8b30-fae68be30955': 0.5}


In [63]:
# BEAT 1 — a teammate corrects it. Only SESSION MEMORY changes here.
await add_feedback(
    SESSION, qa.qa_id,
    feedback_text=(
        # "Prefer the person who reported the bug, not the person who is investigating. "
        # "Veljko reported the bug, so route follow-ups to Veljko first."
        "Prefer the person who can take action, not the reporter. "
        "Milenko said he would take a look, so route follow-ups to Milenko."
    ),
    feedback_score=1,
)
print("— SESSION MEMORY (feedback recorded, not yet applied) —")
await show_session(SESSION)
# note: memify_metadata is now {'feedback_weights_applied': False}

— SESSION MEMORY (feedback recorded, not yet applied) —
  Q: Who should I message next about the node sets bug?
     feedback_text : Prefer the person who can take action, not the reporter. Milenko said he would take a look, so route follow-ups to Milenko.
     feedback_score: 1
     memify_metadata: {'feedback_weights_applied': False}


In [64]:
# BEAT 2 — improve() folds the feedback in. Now BOTH stores change.
#await cognee.improve(dataset=FB_DATASET, session_ids=[SESSION], feedback_alpha=0.8)
from cognee.memify_pipelines.apply_feedback_weights import apply_feedback_weights_pipeline

await apply_feedback_weights_pipeline(
    user=user,
    session_ids=[SESSION],
    dataset=FB_DATASET,
    alpha=0.95,
    run_in_background=False,
)
print("— SESSION MEMORY (applied) —");            await show_session(SESSION)
print("\n— GRAPH weights AFTER —", await show_node_weights(used))
# memify_metadata flips to True; the used nodes' weights drop 0.5 -> 0.1
# (they produced an answer the feedback rated low)


2026-06-08T14:16:23.580606 [info     ] Dataset 7a1a3f18-d0c7-5b22-9f2a-54d470c4d059 is already processed. [cognee.modules.pipelines.layers.check_pipeline_run_qualification]

2026-06-08T14:16:23.591970 [info     ] Pipeline run started: `a66ae156-535f-5235-b15b-cad0fac957bc` [run_tasks_with_telemetry()]

2026-06-08T14:16:23.600151 [info     ] Async Generator task started: `extract_feedback_qas` [run_tasks_base]

2026-06-08T14:16:23.607894 [info     ] Extracting feedback QAs from session cache [extract_feedback_qas]

2026-06-08T14:16:23.608492 [info     ] Coroutine task started: `apply_feedback_weights` [run_tasks_base]

2026-06-08T14:16:23.651009 [info     ] Processed feedback QA e6a16d06-6981-4820-a941-c2ec96ee8741 from session feedback-demo (nodes=9, edges=10, applied=True) [apply_feedback_weights]

2026-06-08T14:16:23.651695 [info     ] Coroutine task completed: `apply_feedback_weights` [run_tasks_base]

2026-06-08T14:16:23.659989 [info     ] Async Generator task completed: `extract_

— SESSION MEMORY (applied) —
  Q: Who should I message next about the node sets bug?
     feedback_text : Prefer the person who can take action, not the reporter. Milenko said he would take a look, so route follow-ups to Milenko.
     feedback_score: 1
     memify_metadata: {'feedback_weights_applied': True}

— GRAPH weights AFTER — {'4d59c4c0-2fb5-5aa0-b756-56e6442f8bbd': 0.025, '1aad5e98-0b66-505d-ad42-1f3282ee478c': 0.025, 'dffcc39f-2b68-59bc-962f-7dcfa5c2c9ba': 0.025, '14e8c4bb-ddcd-5a36-9732-54ee766711ad': 0.025, '073a334a-c403-5b35-ba66-a44d3f5d5774': 0.025, 'e55af088-7425-5cb1-8b30-fae68be30955': 0.025}


In [65]:
# BEAT 3 — re-ask. The feedback now influences retrieval ranking.
after = await cognee.recall(Q, datasets=[FB_DATASET], feedback_influence=1.0)
print("ANSWER (after):", answer(after))
# On a corpus this tiny the top answer may not move yet — but the weights that
# drive future ranking have changed, which is what you just saw in the graph store.


2026-06-08T14:16:26.602417 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='Who should I message next about the node sets bug?' [query_router]

2026-06-08T14:16:27.442808 [info     ] Vector collection retrieval completed: Retrieved distances from 6 collections in 0.82s [cognee.shared.logging_utils]

2026-06-08T14:16:27.443390 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-08T14:16:27.446537 [info     ] ID-filtered retrieval: 25 nodes and 44 edges in 0.00s [cognee.shared.logging_utils]

2026-06-08T14:16:27.447046 [info     ] Graph projection completed: 25 nodes, 44 edges in 0.00s [CogneeGraph]

2026-06-08T14:16:27.447586 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 13, 'connection_count': 10}

2026-06-08T14:16:30.971371 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T14:16:31.027669 [info     ] recall: 1 results across sources=['gr

ANSWER (after): Message Milenko — milenko@topoteretes.com (they said they’ll take a look and can jump on a call).


**What you just saw** — the proof is in the two stores, not the answer text:

- **Beat 1** touched *only* session memory: `feedback_text`/`feedback_score` got
  recorded, but `feedback_weights_applied` was still `False`.
- **Beat 2** is where `improve()` did its work: it flipped that flag to `True` and
  pushed the feedback into the **graph**, dropping the feedback weight on the nodes
  behind the low-rated answer (`0.5 → 0.1`). *That* is the improvement.

The recalled answer may not visibly change on a 3-message corpus (there's no
"Milenko is the expert" fact to surface, and one downvote can't flip a single
candidate). Over real usage these weights re-rank retrieval toward answers people
validated. Knobs: `feedback_score` (1–5), `feedback_alpha`, `feedback_influence`.

## 6. Add another source — Granola meeting notes

The payoff. The bug was *reported* in Slack, but it was *resolved* in a meeting
captured in a Granola note. Because Granola notes normalize to the same
transcript shape, adding them is the same `remember()` call with the same schema.
The note joins the **existing** graph, so a single question can span both sources.

In [17]:
granola_files = sorted(DATA.glob("granola*.txt"))
for f in granola_files:
    await cognee.remember(
        f.read_text(),
        dataset_name=DATASET,
        node_set=["source:granola"],
        graph_model=Thread,
        custom_prompt=EXTRACTION_PROMPT,
    )
    print("remembered", f.name)


2026-06-08T12:47:34.999841 [info     ] Pipeline run started: `7c27735c-08c8-5b61-8f77-299b75644480` [run_tasks_with_telemetry()]

2026-06-08T12:47:35.006076 [info     ] Coroutine task started: `resolve_data_directories` [run_tasks_base]

2026-06-08T12:47:35.012245 [info     ] Coroutine task started: `ingest_data` [run_tasks_base]

2026-06-08T12:47:35.030591 [info     ] Coroutine task completed: `ingest_data` [run_tasks_base]

2026-06-08T12:47:35.036763 [info     ] Coroutine task completed: `resolve_data_directories` [run_tasks_base]

2026-06-08T12:47:35.042832 [info     ] Pipeline run completed: `7c27735c-08c8-5b61-8f77-299b75644480` [run_tasks_with_telemetry()]

2026-06-08T12:47:35.168725 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:47:35.169716 [info     ] No ontology file provided. No owl ontology will be attached to the graph. [OntologyAdapter]

2026-06-08T12:47:35.229432 [info     ] Pipeline run started: `2dce3295-e235-50d8-bb2a-1d

remembered granola.txt


In [18]:
# This answer requires connecting the Slack bug report to the Granola resolution
cross = await cognee.recall(
    "What was the fix for the node sets bug, and where was it explained?",
    datasets=[DATASET],
)
print(answer(cross))

# Node sets work across sources too — scope this query to just the Granola notes:
granola_now = await cognee.recall(
    "What bug was discussed?",
    datasets=[DATASET],
    node_name=["source:granola"],
    scope=["graph"],
)
print("scoped to source:granola ->", answer(granola_now))


2026-06-08T12:48:38.988992 [info     ] query_router: no patterns matched, default=GRAPH_COMPLETION query='What was the fix for the node sets bug, and where was it explained?' [query_router]

2026-06-08T12:48:39.875528 [info     ] Vector collection retrieval completed: Retrieved distances from 14 collections in 0.79s [cognee.shared.logging_utils]

2026-06-08T12:48:39.876338 [info     ] Retrieving ID-filtered graph from database. [CogneeGraph]

2026-06-08T12:48:39.882418 [info     ] ID-filtered retrieval: 47 nodes and 71 edges in 0.01s [cognee.shared.logging_utils]

2026-06-08T12:48:39.883091 [info     ] Graph projection completed: 47 nodes, 71 edges in 0.00s [CogneeGraph]

2026-06-08T12:48:39.883747 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 13, 'connection_count': 10}

2026-06-08T12:48:51.167445 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:48:51.219549 [info     ] recall: 1 results a

Fix: pass node set IDs (not node set names). Explained by Milenko in the Milenko–Veljko chat (Granola note 06f70c5a-7613-4403-a5c9-35ba5821483f) on 2026-05-29.



2026-06-08T12:48:52.053411 [info     ] Vector collection retrieval completed: Retrieved distances from 2 collections in 0.72s [cognee.shared.logging_utils]

2026-06-08T12:48:52.054023 [info     ] Retrieving graph filtered by node type and node name (NodeSet). [CogneeGraph]

2026-06-08T12:48:52.064754 [info     ] Graph projection completed: 3 nodes, 6 edges in 0.00s [CogneeGraph]

2026-06-08T12:48:52.065194 [info     ] Completed resolving edges to text [cognee.shared.logging_utils] extra={'node_count': 3, 'connection_count': 6}

2026-06-08T12:48:54.869475 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]

2026-06-08T12:48:54.917029 [info     ] recall: 1 results across sources=['graph'] (session=-) [recall]


scoped to source:granola -> A bug where using node sets returned/extracted all data because node set names were passed instead of node set IDs — the fix is to pass node set IDs.


## See the graph

`visualize_graph()` writes an interactive HTML view of the whole graph.

In [19]:
await show_graph("graph_typed.html", DATASET)


2026-06-08T12:48:55.020461 [info     ] Retrieved 47 nodes and 71 edges in 0.01 seconds [cognee.shared.logging_utils]

2026-06-08T12:48:55.024127 [info     ] Graph visualization saved as /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/graph_typed.html [cognee.shared.logging_utils]

2026-06-08T12:48:55.024343 [info     ] The HTML file has been stored at path: /Users/milenko/cognee/cognee-companybrain-hackathon/tutorial/graph_typed.html [cognee.shared.logging_utils]

2026-06-08T12:48:55.052048 [info     ] Ladybug database closed successfully [cognee.shared.logging_utils]


## Recap & where to go next

You built a company brain with one call, repeated as you layered on structure:

| Concept | Call |
|---|---|
| Load data + build graph | `cognee.remember(text, dataset_name=...)` |
| Ask questions | `cognee.recall(query, datasets=[...])` |
| Scope recall | `remember(..., node_set=[...])` + `recall(..., node_name=[...], scope=["graph"])` |
| Typed extraction | `remember(..., graph_model=Thread, custom_prompt=...)` |
| Improve / feedback | `add_feedback(...)` + `cognee.improve(dataset=..., feedback_alpha=...)` |
| New source | same `remember()` call, different input |
| Decouple ingest/build | drop to `cognee.add()` + `cognee.cognify(...)` |

From here, the full project (`src/company_brain/`) shows the product version:
live Slack/Granola ingestion, an LLM classifier that auto-tags threads, and a
Slack bot that answers from the graph via `recall()`.